# MeterMind — Model 3: LLM Prompting (Claude)

This notebook implements the **large pretrained LLM baseline** for MeterMind.

Rather than training anything, we simply prompt Claude to rewrite expanded archaic prose into iambic pentameter — zero-shot, using only its pretrained knowledge of poetry and meter.

### Why this is interesting
Claude has been trained on vast amounts of text including poetry, so it has implicit knowledge of iambic meter, Shakespearean register, and stress patterns — knowledge that neither the DP reorderer nor the GRU has access to. The question is whether that broad pretrained knowledge outperforms a task-specific trained model (GRU) or a rule-based symbolic system (DP).

### Pipeline
```
test pairs → Claude API (prompted) → evaluate (MA, SP, G) → compare with Models 1 & 2
```

## 0. Install dependencies

In [ ]:
%pip install anthropic pronouncing sentence-transformers transformers --quiet


## 1. Imports & configuration

In [ ]:
import re, io, os, json, time, math
import pandas as pd
import matplotlib.pyplot as plt
import anthropic
from google.colab import files
from tqdm.notebook import tqdm

# ── Shared utilities ──────────────────────────────────────────────────────────
print('Upload meter_utils.py')
files.upload()
from meter_utils import *

print('Upload eval_metrics.py')
files.upload()
from eval_metrics import *

# ── API ───────────────────────────────────────────────────────────────────────
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", "YOUR_KEY_HERE"))

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL         = "claude-haiku-4-5-20251001"   # swap for claude-sonnet-4-6 for higher quality
MAX_TOKENS    = 100
RETRY_LIMIT   = 3
SLEEP_BETWEEN = 0.3

print('Config ready.')


## 2. Load test set

Upload `training_pairs.csv` — we'll take the same 10% test split as Model 2 for a fair comparison.

**Important:** use the same random seed so the test split is identical across all three models.

In [ ]:
import random
SEED = 42
random.seed(SEED)

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.StringIO(uploaded[filename].decode('utf-8')))
pairs = list(zip(df['input'].tolist(), df['target'].tolist()))
random.shuffle(pairs)

# Replicate exact same split as Model 2.
# Use sorted() to make unique_targets deterministic regardless of Python's
# set ordering — without this, the split changes between runs despite the seed.
unique_targets = sorted({t for _, t in pairs})
random.shuffle(unique_targets)
n = len(unique_targets)
test_targets = set(unique_targets[int(n * 0.9):])
test_pairs   = [(s, t) for s, t in pairs if t in test_targets]

print(f'Test pairs: {len(test_pairs):,}')
print(f'\nExample:')
print(f'  INPUT  : {test_pairs[0][0]}')
print(f'  TARGET : {test_pairs[0][1]}')


## 3. Prompt design

The prompt is the core of this model. It needs to:
1. Tell Claude exactly what iambic pentameter is
2. Constrain it to only use words from the input
3. Preserve archaic register
4. Return only the rewritten line — no explanation

We use a **zero-shot** prompt — no examples provided. This tests Claude's pretrained knowledge directly.

In [ ]:
SYSTEM_PROMPT = """\
You are an expert in Shakespearean poetry and iambic pentameter.

Iambic pentameter is a line of 10 syllables with alternating unstressed and stressed beats:
  da-DUM da-DUM da-DUM da-DUM da-DUM

Your task: rewrite the given archaic prose line into iambic pentameter.

STRICT RULES:
1. Use ONLY words that appear in the input line — do not add new words.
2. You may reorder words freely.
3. Preserve the archaic register (thee, thou, thy, doth, hath, etc.).
4. Aim for exactly 10 syllables matching the da-DUM pattern.
5. Return ONLY the rewritten line — no explanation, no punctuation changes, no quotation marks.
"""

USER_TEMPLATE = "Rewrite this into iambic pentameter:\n{line}"


def prompt_claude(line):
    """Send a single line to Claude and return the rewritten version."""
    for attempt in range(RETRY_LIMIT):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": USER_TEMPLATE.format(line=line)}]
            )
            return response.content[0].text.strip()
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            time.sleep(2 ** attempt)
    return None


# Smoke test
test_out = prompt_claude(test_pairs[0][0])
print(f'INPUT  : {test_pairs[0][0]}')
print(f'TARGET : {test_pairs[0][1]}')
if test_out is None:
    raise RuntimeError("API call failed — check your ANTHROPIC_API_KEY before continuing.")
print(f'OUTPUT : {test_out}')

## 4. Evaluation metrics

Same three metrics as Models 1 and 2.

In [ ]:
# ── Load heavy models before the eval loop ────────────────────────────────────
# meter_utils and eval_metrics were imported in Cell 4.
load_sp_model()
load_gpt2()
print('All metrics ready.')


### 4a. Sample run

Run the full eval pipeline on a small sample before committing to the whole test set — checks the API key, the metrics, and gives you a feel for output quality.

In [ ]:
# ── Sample eval: N_PREVIEW pairs through the full pipeline ───────────────────
N_PREVIEW = 5   # change to see more

_sample_results = []
_sample_failures = []

for _src, _tgt in tqdm(test_pairs[:N_PREVIEW], desc='Sample run'):
    _output = prompt_claude(_src)

    if _output is None:
        print(f"FAILED: {_src}")
        _sample_failures.append(_src)
        continue

    _words = tokenize(_output)
    _ma = metrical_accuracy(_words)
    _sp = semantic_preservation(' '.join(tokenize(_src)), _output)
    _g  = grammaticality(_output)

    # Word constraint check — warn if Claude added words not in input
    _violations = set(tokenize(_output)) - set(tokenize(_src))

    _sample_results.append({
        'input':      _src,
        'target':     _tgt,
        'output':     _output,
        'MA':         round(_ma, 3),
        'SP':         round(_sp, 3),
        'G':          round(_g,  3),
        'violations': sorted(_violations),
    })

    time.sleep(SLEEP_BETWEEN)

print(f"\n{len(_sample_results)} succeeded, {len(_sample_failures)} failed\n")
print(f"{'─'*70}")
for r in _sample_results:
    print(f"INPUT    : {r['input']}")
    print(f"TARGET   : {r['target']}")
    print(f"OUTPUT   : {r['output']}")
    print(f"MA={r['MA']:.3f}  SP={r['SP']:.3f}  G={r['G']:.3f}", end="")
    if r['violations']:
        print(f"  ⚠ added words: {r['violations']}", end="")
    print(f"\n")


## 5. Run on test set

In [ ]:
results  = []
failures = []

for src, tgt in tqdm(test_pairs, desc='Prompting Claude'):
    output = prompt_claude(src)

    if output is None:
        failures.append(src)
        continue

    words = tokenize(output)
    ma = metrical_accuracy(words)
    sp = semantic_preservation(' '.join(tokenize(src)), output)
    g  = grammaticality(output)

    results.append({
        'input':  src,
        'target': tgt,
        'output': output,
        'MA':     round(ma, 4),
        'SP':     round(sp, 4),
        'G':      round(g,  4),
    })

    time.sleep(SLEEP_BETWEEN)

results_df = pd.DataFrame(results)
print(f'Done. {len(results_df):,} results, {len(failures)} failures.')

## 6. Results

In [ ]:
print('=== Claude (LLM) — Test Set Results ===')
print(f"Metrical Accuracy (MA)    : {results_df['MA'].mean():.4f} ± {results_df['MA'].std():.4f}")
print(f"Semantic Preservation (SP): {results_df['SP'].mean():.4f} ± {results_df['SP'].std():.4f}")
print(f"Grammaticality (G)        : {results_df['G'].mean():.4f} ± {results_df['G'].std():.4f}")

print('\n=== Sample outputs ===')
for _, row in results_df.head(5).iterrows():
    print(f"INPUT  : {row['input']}")
    print(f"TARGET : {row['target']}")
    print(f"OUTPUT : {row['output']}")
    print(f"MA={row['MA']:.3f}  SP={row['SP']:.3f}  G={row['G']:.3f}")
    print()

# Distribution plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Claude (LLM) — Metric Distributions', fontsize=14)
for ax, metric, colour in zip(axes, ['MA', 'SP', 'G'], ['steelblue', 'seagreen', 'tomato']):
    ax.hist(results_df[metric], bins=20, color=colour, edgecolor='white', alpha=0.85)
    ax.axvline(results_df[metric].mean(), color='black', linestyle='--', linewidth=1.5,
               label=f"mean={results_df[metric].mean():.3f}")
    ax.set_title(metric)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend()
plt.tight_layout()
plt.savefig('llm_metrics.png', dpi=150)
plt.show()

## 7. Save results

In [ ]:
results_df.to_csv('llm_results.csv', index=False)
print('Saved llm_results.csv')

summary = {
    'model': 'Claude (LLM)',
    'n':     len(results_df),
    'MA':    {'mean': round(float(results_df['MA'].mean()), 4), 'std': round(float(results_df['MA'].std()), 4)},
    'SP':    {'mean': round(float(results_df['SP'].mean()), 4), 'std': round(float(results_df['SP'].std()), 4)},
    'G':     {'mean': round(float(results_df['G'].mean()),  4), 'std': round(float(results_df['G'].std()),  4)},
}
with open('llm_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved llm_summary.json')

files.download('llm_results.csv')
files.download('llm_summary.json')
files.download('llm_metrics.png')